In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("dataset.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df["decision"].value_counts())

(150, 12)
['score', 'decision', 'age', 'baseline_score', 'comorbidity_ratio', 'dependants', 'prior_visits', 'recent_admissions', 'requested_beds', 'vitals_index', 'ward', 'years_registered']
    score decision     age  baseline_score  comorbidity_ratio  dependants  \
0  0.9829  APPROVE  72.733         812.000              0.013           1   
1  0.9827  APPROVE  72.733         812.555              0.013           1   
2  0.9829  APPROVE  72.733         812.055              0.013           1   
3  0.9829  APPROVE  72.733         812.000              0.013           1   
4  0.6612  APPROVE  46.500         600.000              0.500           3   

   prior_visits  recent_admissions  requested_beds  vitals_index ward  \
0          20.0              0.554             0.0             0    B   
1          20.0              0.554             0.0             0    B   
2          20.0              0.554             0.0             0    B   
3          20.0              0.554             0.0    

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

num = ["age","baseline_score","comorbidity_ratio","dependants","prior_visits",
       "recent_admissions","requested_beds","vitals_index","years_registered"]

train = pd.read_csv("dataset.csv").drop_duplicates(subset=num+["ward"])   # Round 1 + 2 queries
test  = pd.read_csv("new_data.csv")                                        # Round 4 queries = UNSEEN
print("train rows:", len(train), "| unseen test rows:", len(test))

def prep(d):
    X = d[num].copy()
    for w in "ABCD": X[f"ward_{w}"] = (d["ward"] == w).astype(float)
    return X.astype(float)

models = {
  "Linear":            make_pipeline(StandardScaler(), Ridge(alpha=10)),
  "Poly2+Ridge":       make_pipeline(StandardScaler(), PolynomialFeatures(2), RidgeCV(alphas=np.logspace(0,3,13))),
  "GradientBoosting":  GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=0),
  "RandomForest":      RandomForestRegressor(n_estimators=300, random_state=0),
}
CUTOFF = 0.45
for name, m in models.items():
    m.fit(prep(train), train["score"])
    p = np.clip(m.predict(prep(test)), 0, 1)
    dec = np.where(p >= CUTOFF, "APPROVE", "DECLINE")
    mae = np.abs(p - test["score"]).mean()
    r2 = 1 - ((p-test["score"])**2).sum() / ((test["score"]-test["score"].mean())**2).sum()
    print(f"{name:17s} MAE={mae:.4f}  R2={r2:.2f}  decision accuracy={(dec==test['decision']).mean():.2f}")

final = models["Poly2+Ridge"]

train rows: 135 | unseen test rows: 37
Linear            MAE=0.1433  R2=0.37  decision accuracy=0.70
Poly2+Ridge       MAE=0.0959  R2=0.66  decision accuracy=0.86
GradientBoosting  MAE=0.1972  R2=-0.13  decision accuracy=0.68
RandomForest      MAE=0.1985  R2=-0.14  decision accuracy=0.68


Trained on Round 1 and 2 queries only (dataset.csv, 135 unique rows) and evaluated on the 37 Round 4 queries (new_data.csv), which were never used for training. Poly2+Ridge generalised best (MAE 0.096, R² 0.66, decision accuracy 0.86), while tree models (Gradient Boosting, Random Forest) failed to extrapolate beyond the narrow range of the training data (MAE about 0.20). The decision cutoff of 0.45 lies between the highest DECLINE score (0.4408) and the lowest APPROVE score (0.4605) observed, and was set using the Round 4 queries.